# Key observations thus far: 

## Conclusions from a niave approach: 
- Counting the number of opperations comparing string simiarlity of each item with every other results in $\frac{n(n-1)}{2}$ the testing dataset ts consists of 1,000,000 rows and so one would expect to peform ≈ <span style="color:red">500 billion opperations</span> <br>
- **Experements have estimated around Comparisions/sec = 21,214.  One would thus expect to wait for the process to complete for 6547 hours. Not a practical approach.** 

<br><br>
The severity of this situation can be remedied slightly by deciding a effective blocking stratergy.  For example: <br> <br>

In the testing set,  The class balance is roughly split evenly between US and India adresses. (59% US and 40% India [other countrys in testing]) <br>
Splitting 1,000,000 the data down the midde, to subsets of n(n-1/2)  => (500,000*500,000)/2 *2 => <span style="color:red">250 billion opperations.</span>  ie half time. 

> $n^2$ scaling seems pretty bad, but other other methods may also not scale linearly or better so suggests to potential value of considering blocking stratergy

## Possible blocking stratergys

### Investigate the Proportions of other contires and prevelence of diffrent geographic terms

In [6]:
import sys
!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m spacy download en_core_web_sm

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 27.2 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 26.2.2
    Uninstalling pip-26.2.2:
      Successfully uninstalled pip-26.2.2
Defaulting to user installation because normal site-packages is not writeable
  Using cached en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
import matplotlib.pyplot as plt
from utils.import_data import ImportData
import pandas as pd

# ==== Spacey NER for Geopolitical entities to help with india, france etc vibe code experement =====
import spacy
from collections import Counter
from tqdm.auto import tqdm



C:\Users\Graem\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')
C:\Users\Graem\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
ts = ImportData().import_training_data()
test_s = ImportData().import_testing_data()
ts.info()

# Learn about the proportions of countries in the training set
print("\nTraining set Countries")
counts = ts["country"].value_counts()
result = pd.DataFrame({
    "count": counts,
    "pct": counts / counts.sum() * 100
})
print(result)

# Learn about the proportions of countries in the testing set 
print("\nTesting set Countries")
counts = test_s["country"].value_counts()
result = pd.DataFrame({
    "count": counts,
    "pct": counts / counts.sum() * 100
})
print(result)

# Get a flavour for how Adresses look
USA_India = pd.concat([ts[ts["country"]=="US"].head(10), ts[ts["country"]=="India"].head(10),])[["business_address","country"]]
India_adr = USA_India[USA_India["country"]=="India"]["business_address"]

for adr in enumerate(India_adr):
    print(adr)


# Only load components we need
nlp = spacy.load(
    "en_core_web_sm",
    disable=[
        "tagger",
        "parser",
        "lemmatizer",
        "attribute_ruler"
    ]
)

texts = ts["business_address"].fillna("").astype(str)

gpe_counter = Counter()

for i, doc in enumerate(
    tqdm(
        nlp.pipe(
            texts,
            batch_size=5000
        ),
        total=len(texts),
        desc="Processing rows"
    )
):
    gpe_counter.update(
        ent.text.strip().upper()
        for ent in doc.ents
        if ent.label_ == "GPE"
    )

    # Progress report every 100k rows
    if (i + 1) % 100_000 == 0:
        print(f"\nProcessed {i+1:,} rows")
        print("Top GPEs so far:")
        print(gpe_counter.most_common(10))

        # Optional checkpoint
        (
            pd.DataFrame(
                gpe_counter.items(),
                columns=["GPE", "count"]
            )
            .sort_values("count", ascending=False)
            .to_csv(
                f"gpe_counts_{i+1}.csv",
                index=False
            )
        )

# Final output
gpe_counts = (
    pd.DataFrame(
        gpe_counter.items(),
        columns=["GPE", "count"]
    )
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

print(gpe_counts.head(20))

``` c
<class 'pandas.DataFrame'>
RangeIndex: 12527040 entries, 0 to 12527039
Data columns (total 4 columns):
 #   Column            Dtype
---  ------            -----
 0   entity_id         str  
 1   business_name     str  
 2   business_address  str  
 3   country           str  
dtypes: str(4)
memory usage: 382.3 MB

Training set Countries
           count        pct
country                    
US       7510506  59.954355
India    5016534  40.045645

Testing set Countries
           count        pct
country                    
India    5527551  47.235414
US       4480137  38.284790
France   1694445  14.479796
(0, '797, Lake Town Block A, Kolkata, Howrah, West Bengal')
(1, '2505, Tower 1, Oakwood, Runwal Greens, Mulund Goreagon Link Road, Near Fortis Hospital, Bhandup West, Mumbai, Maharashtra')
(2, 'H.No.16-11-23/37/A, 2Nd Floor, Flat No.207, Sagar Hotel Building, Opp.Rta Office, Mo, Osarambagh, Hyderabad, Telangana')
(3, '303, 3Rd Floor Sakar 5 B/H Natraj Cinema Ashram Road, Ahmedabad, Gujarat')
(4, 'House No-777 Raghubir Bhawanmadan Pur Khadar Sarita Vihar, Delhi, South Delhi, Delhi')
(5, 'Secunderabad, Hyderabad, Plot No.53, Flat No.402, Sr Estate-I Rainbow Colony, Ammuguda, Sainikpuri, Telangana, # 53/1')
(6, 'E-7, Second Floor, New Delhi, South Delhi, Delhi')
(7, '46/1106, Dhanalakshmi Road, Near Amani Auditorium, Kannur, Kerala')
(8, 'Building No. 37/4, First Floor Opp Bharata Mata College, Vazhakkala, Thrikkakara, Ernakulam, Kerala')
(9, '#17/1, Lower Ground Floor, 1St Cross, Lalbagh Road, Bangalore South, Bangalore, Karnataka')
Processing rows:   1%|          | 95001/12527040 [02:31<4:02:16, 855.21it/s] 
Processed 100,000 rows
Top GPEs so far:
[('DELHI', 7176), ('NC', 4195), ('MUMBAI', 3758), ('NEW DELHI', 3659), ('KARNATAKA', 3040), ('VA', 2752), ('GUJARAT', 2653), ('TELANGANA', 2554), ('WEST BENGAL', 2408), ('KOLKATA', 2305)]
Processing rows:   2%|▏         | 200000/12527040 [04:32<3:16:18, 1046.55it/s]
Processed 200,000 rows
Top GPEs so far:
[('DELHI', 14307), ('NC', 8372), ('MUMBAI', 7489), ('NEW DELHI', 7266), ('KARNATAKA', 6222), ('VA', 5350), ('GUJARAT', 5264), ('TELANGANA', 5003), ('WEST BENGAL', 4876), ('KOLKATA', 4629)]
Processing rows:   2%|▏         | 201483/12527040 [04:41<5:06:34, 670.07it/s]
```